# Plant Instance Wirings — Notebook

End-to-end walkthrough of the two plant-instance retrieval wirings:

- **Plant→Image** (`emb_plant2image.json`) — given a per-plant crop instance, retrieve full field images that contain the same or a similar plant.
- **Plant→Plant** (`emb_plant2plant.json`) — retrieve other instances of the same species using `instance_labels`.

Both wirings convert to the same `MetadataGroup` representation used by Image→Image, so all KPIs are available including graded `knn_metadata_ndcg`.

Run all cells top-to-bottom; all figures are interactive (Plotly).

In [8]:
import json
import sys
from collections import Counter
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from precisionai.agrieval.emb.services.evaluate import (
    run_plant2image_eval,
    run_plant2plant_eval,
)
from precisionai.agrieval.emb.services.reporting import (
    plot_cosine_similarity,
    plot_knn_confusion,
    plot_tsne,
    print_result,
)

---
## Part 1 — Plant→Image

`emb_plant2image.json` contains:
- **20 parent full-field images** across four L2 clusters (A1, A2, D1, D2) — 5 images per cluster.
- **52 instance crops** drawn from those parents (up to 3 per image).
- **72 embeddings total** across two crop classes (A and D).

Parent images are stored under `images/` and instance crops under `instances/`.  
The `instance_to_image` mapping declares which crops belong to which parent.

Ground truth:
- Instance → its parent is grade-3 (explicit positive).
- Parent → all its instances are grade-3.
- Items sharing the same crop class (A or D) across different parent groups are grade-2.

**Similarity design** — instances of the same parent image have cosine ≈ 0.97–0.99 to their parent (very close, not identical). Parent images of the same class are 0.80–0.95 apart. Cross-class (A vs D) is near zero.

### Load Data

In [9]:
with open("emb_plant2image.json") as f:
    payload = json.load(f)

embeddings = payload["embeddings"]
instance_to_image = payload["instance_to_image"]

print(f"Total embeddings : {len(embeddings)}")
print(f"Parent images    : {len(instance_to_image)}")
print(f"Instance crops   : {sum(len(v) for v in instance_to_image.values())}")
print(f"Embedding dim    : {len(next(iter(embeddings.values())))}")
print()
print("Sample parent->instances mapping:")
for parent, insts in list(instance_to_image.items())[:2]:
    print(f"  {parent}")
    for i in insts:
        print(f"    -> {i}")

Total embeddings : 72
Parent images    : 20
Instance crops   : 52
Embedding dim    : 32

Sample parent->instances mapping:
  images/A1/pai-1NY5xYvV.png
    -> instances/A1/pai-1NY5xYvV-1.png
    -> instances/A1/pai-1NY5xYvV-2.png
    -> instances/A1/pai-1NY5xYvV-3.png
  images/A1/pai-3Dab9LlQ.png
    -> instances/A1/pai-3Dab9LlQ-2.png
    -> instances/A1/pai-3Dab9LlQ-3.png
    -> instances/A1/pai-3Dab9LlQ-4.png


### Run Evaluation

In [10]:
result_p2i = run_plant2image_eval(
    embeddings=embeddings,
    instance_to_image=instance_to_image,
    k_values=[5, 10],
    dataset_root="images",
    sample_pairs=None,
)
print_result(result_p2i)

n_items      : 72
embedding_dim: 32
classes      : ['A', 'D']
k_values     : [5, 10]

── global_metrics ──────────────────────────────────────────────────────
  pairwise cosine    : mean=0.1960  std=0.4117  (p05=-0.3448  p50=0.0603  p95=0.8985)
  centroid cosine    : mean=0.4551  std=0.1396  norm=0.4551
  intra/inter gap    : 0.5131  (intra=0.4529  inter=-0.0602)
  effective_rank     : 3.74  (ratio=0.1170  dim=32)
  uniformity         : -1.8269
  alignment          : 0.0145

  hubness@5         : mean=5.0000  std=3.3250  p95=10.4500
  hubness@10        : mean=10.0000  std=5.3385  p95=19.0000
  knn_radius@5         : mean=0.8719  std=0.0301  p05=0.8142  p95=0.9108
  knn_radius@10        : mean=0.8416  std=0.0283  p05=0.7920  p95=0.8785
  mean_top_k_sim@5         : mean=0.9428  std=0.0179  p05=0.9181  p95=0.9617
  mean_top_k_sim@10        : mean=0.8974  std=0.0208  p05=0.8569  p95=0.9226
  outlier_score@5         : mean=0.0572  std=0.0179  p95=0.0819
  outlier_score@10        : mean=0.10

### Interpreting the Plant→Image KPIs

| KPI | What to look for |
|---|---|
| `knn_metadata_precision@k` | Are the top-k results explicit positives (parent/instances from the same group)? |
| `knn_metadata_ndcg@k` | Is the full grade-3 set (parent + instances) ranked above grade-2 (same class, different group)? |
| `knn_label_purity@k` | Fraction of neighbours sharing the same crop class (A or D). |
| `alignment` | Mean ‖u−v‖² across explicit positive pairs — lower means parent and instances are closer together. |

With the tight embeddings in this file, you should see **high metadata precision and nDCG** because instances are only 0.97–0.99 cosine away from their parent, while cross-class items are near zero.

### Visualizations

All plots are interactive — hover for details, click legend entries to toggle classes.

In [11]:
plot_knn_confusion(result_p2i, output_path=None)
plot_cosine_similarity(embeddings, result_p2i, output_path=None)
plot_tsne(embeddings, result_p2i, dimensions=2, output_path=None)

  t-SNE 2D — fitting 72 samples (perplexity=12, iter=1000)...


---
## Part 2 — Plant→Plant

`emb_plant2plant.json` contains 52 instance crops across three species labels:
- **Crop | Soybean** — 49 instances (A1: 13, A2: 14, D1: 12, D2: 10)
- **Weed | Water Hemp** — 2 instances (A1)
- **Weed | Grass** — 1 instance (A2)

All instance crops are stored under `instances/`. All instances sharing the same class label are mutual grade-3 positives — useful for measuring class-level retrieval quality.

### Load Data

In [12]:
with open("emb_plant2plant.json") as f:
    payload = json.load(f)

embeddings_p2p = payload["embeddings"]
instance_labels = payload["instance_labels"]

label_counts = Counter(instance_labels.values())
print(f"Total instances   : {len(embeddings_p2p)}")
print(f"Class distribution: {dict(label_counts)}")

Total instances   : 52
Class distribution: {'Crop | Soybean': 49, 'Weed | Water Hemp': 2, 'Weed | Grass': 1}


### Run Evaluation

In [13]:
result_p2p = run_plant2plant_eval(
    embeddings=embeddings_p2p,
    instance_labels=instance_labels,
    k_values=[5, 10],
    sample_pairs=None,
)
print("=== Plant→Plant ===")
print_result(result_p2p)

=== Plant→Plant ===
n_items      : 52
embedding_dim: 32
classes      : ['Crop | Soybean', 'Weed | Grass', 'Weed | Water Hemp']
k_values     : [5, 10]

── global_metrics ──────────────────────────────────────────────────────
  pairwise cosine    : mean=0.7457  std=0.2522  (p05=0.0320  p50=0.8302  p95=0.8964)
  centroid cosine    : mean=0.8664  std=0.1889  norm=0.8664
  intra/inter gap    : 0.7766  (intra=0.8330  inter=0.0565)
  effective_rank     : 6.21  (ratio=0.1941  dim=32)
  uniformity         : -0.7646
  alignment          : 0.3340

  hubness@5         : mean=5.0000  std=4.9225  p95=13.9000
  hubness@10        : mean=10.0000  std=8.5124  p95=23.9000
  knn_radius@5         : mean=0.8406  std=0.1580  p05=0.5527  p95=0.9090
  knn_radius@10        : mean=0.8226  std=0.1732  p05=0.5147  p95=0.8937
  mean_top_k_sim@5         : mean=0.8653  std=0.1008  p05=0.6769  p95=0.9168
  mean_top_k_sim@10        : mean=0.8473  std=0.1342  p05=0.6014  p95=0.9068
  outlier_score@5         : mean=0.134

### Visualizations

All plots are interactive — hover for details, click legend entries to toggle classes.

In [14]:
plot_knn_confusion(result_p2p, output_path=None)
plot_cosine_similarity(embeddings_p2p, result_p2p, output_path=None)
plot_tsne(embeddings_p2p, result_p2p, dimensions=2, output_path=None)

  t-SNE 2D — fitting 52 samples (perplexity=8, iter=1000)...
